# Indic-Transcribe-core GPU pilot

Controlled, non-production comparison of `bodhan-ai/indic-transcribe-core` with faster-whisper Base on three Kathbath samples.

**Run only in a GPU-backed Google Colab runtime. Do not run this notebook on the local Windows machine.**

Pinned revision: `e083ab7dfeeddcb5c0b4a2d7df53f53e7b0f1537`

Accept the gated model terms on Hugging Face and add a Colab Secret named `HF_TOKEN`. The token is never printed. Upload only `v1.json` and the three selected WAV files.

In [ ]:
import platform
import sys
import time
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. Select a Colab GPU runtime; do not run locally.")
gpu = torch.cuda.get_device_properties(0)
runtime = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": gpu.name,
    "gpu_total_memory_gib": round(gpu.total_memory / 2**30, 2),
    "gpu_free_memory_gib": round(torch.cuda.mem_get_info()[0] / 2**30, 2),
}
runtime

In [ ]:
# Experimental-only dependencies; project requirements are not changed.
%pip install -q torch torchaudio transformers sentencepiece soundfile huggingface_hub faster-whisper==1.1.1 "av<19"

In [ ]:
from google.colab import files

uploaded = files.upload()
required = {
    "v1.json",
    "kathbath-hi-01-row-200.wav",
    "kathbath-te-01-row-1000.wav",
    "kathbath-ml-01-row-400.wav",
}
missing = required.difference(uploaded)
if missing:
    raise FileNotFoundError(f"Required files missing: {sorted(missing)}")
print(f"Uploaded {len(uploaded)} files; required files are present.")

In [ ]:
import hashlib
import json

manifest = json.loads(Path("v1.json").read_text(encoding="utf-8"))
target_ids = {
    "indic-asr-eval-kathbath-hi-01",
    "indic-asr-eval-kathbath-te-01",
    "indic-asr-eval-kathbath-ml-01",
}
samples = [sample for sample in manifest["samples"] if sample["id"] in target_ids]
if {sample["id"] for sample in samples} != target_ids:
    raise ValueError("The uploaded manifest does not contain exactly the target samples.")

for sample in samples:
    audio_path = Path(sample["audio_path"]).name
    if not audio_path.is_file():
        raise FileNotFoundError(audio_path)
    digest = hashlib.sha256(audio_path.read_bytes()).hexdigest()
    if digest != sample["checksum_sha256"]:
        raise ValueError(f"Checksum mismatch for {sample['id']}")
    sample["colab_audio_path"] = str(audio_path)

for sample in samples:
    print(sample["id"], sample["language"], sample["colab_audio_path"])


In [ ]:
import unicodedata

def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", text).casefold()
    cleaned = []
    for character in normalized:
        category = unicodedata.category(character)
        if character.isspace():
            cleaned.append(" ")
        elif category.startswith(("P", "S")):
            cleaned.append(" ")
        else:
            cleaned.append(character)
    return " ".join("".join(cleaned).split())

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for row, reference_token in enumerate(reference, 1):
        current = [row]
        for column, hypothesis_token in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[column] + 1, previous[column - 1] + (reference_token != hypothesis_token)))
        previous = current
    return previous[-1]

def wer(reference, hypothesis):
    reference_tokens = normalize_text(reference).split()
    hypothesis_tokens = normalize_text(hypothesis).split()
    if not reference_tokens:
        return 0.0 if not hypothesis_tokens else 1.0
    return edit_distance(reference_tokens, hypothesis_tokens) / len(reference_tokens)

def cer(reference, hypothesis):
    reference_chars = list(normalize_text(reference).replace(" ", ""))
    hypothesis_chars = list(normalize_text(hypothesis).replace(" ", ""))
    if not reference_chars:
        return 0.0 if not hypothesis_chars else 1.0
    return edit_distance(reference_chars, hypothesis_chars) / len(reference_chars)

def script_distribution(text):
    ranges = {
        "Devanagari": ("\u0900", "\u097f"),
        "Telugu": ("\u0c00", "\u0c7f"),
        "Malayalam": ("\u0d00", "\u0d7f"),
        "Tamil": ("\u0b80", "\u0bff"),
        "Arabic": ("\u0600", "\u06ff"),
        "Latin": ("A", "z"),
    }
    counts = {name: 0 for name in ranges}
    counts["Other"] = 0
    for character in text:
        if character.isspace() or unicodedata.category(character).startswith(("P", "N")):
            continue
        for name, (start, end) in ranges.items():
            if start <= character <= end:
                counts[name] += 1
                break
        else:
            counts["Other"] += 1
    return {key: value for key, value in counts.items() if value}


In [ ]:
from google.colab import userdata
from huggingface_hub import snapshot_download

MODEL_ID = "bodhan-ai/indic-transcribe-core"
MODEL_REVISION = "e083ab7dfeeddcb5c0b4a2d7df53f53e7b0f1537"
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Colab Secret HF_TOKEN is missing; accept the gated model terms first.")

load_started = time.perf_counter()
try:
    model_dir = snapshot_download(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=HF_TOKEN,
        allow_patterns=["*.py", "*.json", "*.model", "*.safetensors", "*license*", "*.md"],
    )
    model_size_bytes = sum(path.stat().st_size for path in Path(model_dir).rglob("*") if path.is_file())
    import sys
    sys.path.insert(0, model_dir)
    from indic_transcribe import IndicTranscribe
    asr = IndicTranscribe.from_pretrained(model_dir)
    model_load_seconds = time.perf_counter() - load_started
except Exception as exc:
    error_record = {
        "status": "model_load_error",
        "model": MODEL_ID,
        "revision": MODEL_REVISION,
        "error": repr(exc),
        "runtime": runtime,
    }
    Path("indic-transcribe-core-pilot.json").write_text(json.dumps(error_record, indent=2, ensure_ascii=False), encoding="utf-8")
    raise

model_metadata = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "download_size_bytes": model_size_bytes,
    "model_load_seconds": model_load_seconds,
    "runtime": runtime,
}
model_metadata

In [ ]:
core_results = []
for sample in samples:
    started = time.perf_counter()
    try:
        transcription, lid = asr.transcribe(sample["colab_audio_path"], lang=sample["language"], return_lid=True)
        core_results.append({
            "sample_id": sample["id"],
            "language": sample["language"],
            "model_revision": MODEL_REVISION,
            "selected_language": sample["language"],
            "language_identification": lid,
            "reference": sample["reference_transcript"],
            "hypothesis": transcription,
            "script_distribution": script_distribution(transcription),
            "inference_latency_seconds": time.perf_counter() - started,
            "wer": wer(sample["reference_transcript"], transcription),
            "cer": cer(sample["reference_transcript"], transcription),
            "status": "ok",
        })
    except Exception as exc:
        core_results.append({
            "sample_id": sample["id"],
            "language": sample["language"],
            "model_revision": MODEL_REVISION,
            "selected_language": sample["language"],
            "status": "error",
            "error": repr(exc),
        })
core_results

In [ ]:
from faster_whisper import WhisperModel

BASE_MODEL_REVISION = "ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66"
base_load_started = time.perf_counter()
base_model = WhisperModel("base", device="cpu", compute_type="int8")
base_load_seconds = time.perf_counter() - base_load_started
base_results = []
for sample in samples:
    started = time.perf_counter()
    try:
        segments, info = base_model.transcribe(
            sample["colab_audio_path"],
            language=sample["language"],
            task="transcribe",
            vad_filter=True,
            vad_parameters={"min_silence_duration_ms": 700, "speech_pad_ms": 300},
            beam_size=1,
            condition_on_previous_text=False,
        )
        text = " ".join(segment.text.strip() for segment in segments if segment.text.strip())
        base_results.append({
            "sample_id": sample["id"],
            "language": sample["language"],
            "model_revision": BASE_MODEL_REVISION,
            "selected_language": sample["language"],
            "detected_language": info.language,
            "reference": sample["reference_transcript"],
            "hypothesis": text,
            "script_distribution": script_distribution(text),
            "inference_latency_seconds": time.perf_counter() - started,
            "wer": wer(sample["reference_transcript"], text),
            "cer": cer(sample["reference_transcript"], text),
            "status": "ok",
        })
    except Exception as exc:
        base_results.append({"sample_id": sample["id"], "language": sample["language"], "status": "error", "error": repr(exc)})
base_results

In [ ]:
import csv
from statistics import mean

def summarize(rows):
    successful = [row for row in rows if row.get("status") == "ok"]
    languages = sorted({row["language"] for row in successful})
    return {
        "total": len(rows),
        "successful": len(successful),
        "failed": len(rows) - len(successful),
        "overall_wer": mean(row["wer"] for row in successful) if successful else None,
        "overall_cer": mean(row["cer"] for row in successful) if successful else None,
        "by_language": {
            language: {
                "wer": mean(row["wer"] for row in successful if row["language"] == language),
                "cer": mean(row["cer"] for row in successful if row["language"] == language),
            }
            for language in languages
        },
    }

pilot = {
    "status": "complete" if all(row.get("status") == "ok" for row in core_results) else "partial_failure",
    "model": model_metadata,
    "core_summary": summarize(core_results),
    "core_results": core_results,
    "baseline": {
        "model": "Systran/faster-whisper-base",
        "revision": BASE_MODEL_REVISION,
        "device": "cpu",
        "compute_type": "int8",
        "model_load_seconds": base_load_seconds,
        "summary": summarize(base_results),
        "results": base_results,
    },
    "limitations": [
        "Three samples are a smoke-test pilot, not model-replacement evidence.",
        "Do not compare directly with the model card OIWER benchmark.",
    ],
}
Path("indic-transcribe-core-pilot.json").write_text(json.dumps(pilot, indent=2, ensure_ascii=False), encoding="utf-8")
rows = [{"model": "indic-transcribe-core", **row} for row in core_results]
rows += [{"model": "faster-whisper-base", **row} for row in base_results]
fields = sorted({key for row in rows for key in row})
with Path("indic-transcribe-core-pilot.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
print(json.dumps(pilot["core_summary"], indent=2, ensure_ascii=False))
print("Saved indic-transcribe-core-pilot.json and indic-transcribe-core-pilot.csv")

## Optional 15-sample expansion gate

Run the expansion only after reviewing the three-sample artifacts and confirming that all three Core samples completed successfully. Upload the remaining 12 WAV files, apply the same checksum and explicit-language gates, and save the expanded run as `indic-transcribe-core-full-15.json` and `indic-transcribe-core-full-15.csv`. Never overwrite historical or pilot artifacts.